# Vision Desk — Colab lab

**Course:** Master's Finance, Data & AI — Computer Vision in Finance  
**Repo:** [LikhitaYerra/vision-desk](https://github.com/LikhitaYerra/vision-desk)  
**Case:** Vision Desk (ViT → JSON → MCP → grounded LLM)

## Mission
1. Build a tiny synthetic finance-image dataset (chart types + page types)
2. Split **by document**, not by page
3. Fine-tune a **ViT** (head first) and beat a majority baseline
4. Emit a **JSON schema** from vision
5. Expose four MCP-style tools and show a **grounded** answer + a **refusal**

Runtime: GPU recommended (T4). CPU works but slower.


## 0. Install


In [ ]:
%pip install -q timm scikit-learn matplotlib pillow


## 1. Imports & config


In [ ]:
from __future__ import annotations

import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import timm
from PIL import Image, ImageDraw, ImageFont
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
IMG_SIZE = 224
BATCH = 16
EPOCHS_HEAD = 8
LR_HEAD = 1e-3
N_DOCS_PER_CLASS = 8   # documents
PAGES_PER_DOC = 4      # pages/images per document
VAL_DOC_FRAC = 0.25

CLASSES = ["candlestick_chart", "line_chart", "page_balance_sheet", "page_notes"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("vision_desk_lab")
DATA = ROOT / "images"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
for p in (DATA, OUT, FIG):
    p.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("classes:", CLASSES)


## 2. Synthetic finance images

We draw simple charts / pages so the lab runs without an external dataset.
Each image belongs to a **document id** — used for the split.


In [ ]:
def _font(size=18):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size)
    except Exception:
        return ImageFont.load_default()


def draw_candlestick(path: Path, seed: int, ticker: str | None = "AAPL"):
    rng = random.Random(seed)
    img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (248, 250, 252))
    d = ImageDraw.Draw(img)
    d.rectangle((20, 30, 200, 200), outline=(45, 107, 232), width=2)
    x = 35
    for _ in range(10):
        top = rng.randint(50, 140)
        h = rng.randint(20, 50)
        color = (45, 171, 106) if rng.random() > 0.45 else (217, 79, 79)
        d.line((x + 6, top - 8, x + 6, top + h + 8), fill=(28, 38, 59), width=2)
        d.rectangle((x, top, x + 12, top + h), fill=color)
        x += 16
    label = f"{ticker or '????'} · candlestick"
    d.text((24, 8), label, fill=(28, 38, 59), font=_font(14))
    img.save(path)


def draw_line(path: Path, seed: int, ticker: str | None = "MSFT"):
    rng = random.Random(seed)
    img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (248, 250, 252))
    d = ImageDraw.Draw(img)
    d.rectangle((20, 30, 200, 200), outline=(45, 107, 232), width=2)
    pts = []
    y = rng.randint(80, 150)
    for i in range(12):
        y = min(190, max(40, y + rng.randint(-18, 18)))
        pts.append((30 + i * 14, y))
    d.line(pts, fill=(45, 107, 232), width=3)
    d.text((24, 8), f"{ticker or '????'} · line", fill=(28, 38, 59), font=_font(14))
    img.save(path)


def draw_bilan(path: Path, seed: int, ticker: str | None = "AAPL"):
    img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (255, 255, 255))
    d = ImageDraw.Draw(img)
    d.rectangle((16, 16, 208, 48), fill=(31, 166, 158))
    d.text((28, 24), "BILAN", fill=(255, 255, 255), font=_font(18))
    for i in range(7):
        y = 64 + i * 18
        fill = (241, 245, 250) if i % 2 == 0 else (255, 255, 255)
        d.rectangle((16, y, 208, y + 16), fill=fill, outline=(200, 210, 220))
    d.text((28, 200), f"issuer={ticker or 'UNKNOWN'}", fill=(61, 71, 92), font=_font(12))
    img.save(path)


def draw_notes(path: Path, seed: int, ticker: str | None = None):
    img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (255, 255, 255))
    d = ImageDraw.Draw(img)
    d.text((24, 20), "Notes annexes", fill=(19, 28, 48), font=_font(16))
    for i in range(10):
        d.line((24, 55 + i * 14, 200, 55 + i * 14), fill=(180, 190, 200), width=2)
    if ticker:
        d.text((24, 200), f"ref={ticker}", fill=(61, 71, 92), font=_font(12))
    else:
        d.text((24, 200), "ref=????", fill=(217, 79, 79), font=_font(12))
    img.save(path)


DRAWERS: dict[str, Callable] = {
    "candlestick_chart": draw_candlestick,
    "line_chart": draw_line,
    "page_balance_sheet": draw_bilan,
    "page_notes": draw_notes,
}


@dataclass
class Sample:
    path: Path
    label: str
    doc_id: str
    ticker: str | None


def build_dataset() -> list[Sample]:
    samples: list[Sample] = []
    tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", None]  # None → OCR failure case
    n = 0
    for cls in CLASSES:
        for d in range(N_DOCS_PER_CLASS):
            doc_id = f"{cls}__doc{d:02d}"
            for p in range(PAGES_PER_DOC):
                ticker = tickers[(d + p) % len(tickers)]
                # force some null tickers on held-out-looking docs
                if d >= int(N_DOCS_PER_CLASS * (1 - VAL_DOC_FRAC)) and p == 0:
                    ticker = None
                fname = DATA / f"{doc_id}_p{p}.png"
                DRAWERS[cls](fname, seed=1000 + n, ticker=ticker)
                samples.append(Sample(fname, cls, doc_id, ticker))
                n += 1
    return samples


samples = build_dataset()
print(f"generated {len(samples)} images in {DATA}")
print("example:", samples[0])


## 3. Preview a few images


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
shown = {}
for s in samples:
    if s.label not in shown:
        shown[s.label] = s
    if len(shown) == 4:
        break
for ax, cls in zip(axes, CLASSES):
    ax.imshow(Image.open(shown[cls].path))
    ax.set_title(cls.replace("_", "\n"), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG / "preview_classes.png", dpi=120)
plt.show()


## 4. Document-level split

Same PDF / issuer never appears in both train and test.


In [ ]:
def split_by_document(samples: list[Sample], val_frac=VAL_DOC_FRAC, seed=SEED):
    by_doc: dict[str, list[Sample]] = {}
    for s in samples:
        by_doc.setdefault(s.doc_id, []).append(s)
    docs = sorted(by_doc)
    rng = random.Random(seed)
    rng.shuffle(docs)
    n_val = max(1, int(len(docs) * val_frac))
    val_docs = set(docs[:n_val])
    train, val = [], []
    for doc, items in by_doc.items():
        (val if doc in val_docs else train).extend(items)
    return train, val, val_docs


train_s, val_s, val_docs = split_by_document(samples)
print(f"train images={len(train_s)} | val images={len(val_s)} | val docs={len(val_docs)}")
assert not ({s.doc_id for s in train_s} & {s.doc_id for s in val_s}), "document leakage!"
print("no document leakage ✓")


## 5. Dataset + majority baseline


In [ ]:
tf_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
tf_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class VisionDeskDataset(Dataset):
    def __init__(self, items: list[Sample], train: bool):
        self.items = items
        self.tf = tf_train if train else tf_eval

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        s = self.items[idx]
        x = self.tf(Image.open(s.path).convert("RGB"))
        y = CLASS_TO_ID[s.label]
        return x, y, s.path.name, s.doc_id, s.ticker or ""


train_loader = DataLoader(VisionDeskDataset(train_s, True), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(VisionDeskDataset(val_s, False), batch_size=BATCH)

# Majority baseline on val
from collections import Counter
maj = Counter(s.label for s in train_s).most_common(1)[0][0]
y_true = [s.label for s in val_s]
y_maj = [maj] * len(val_s)
f1_maj = f1_score(y_true, y_maj, average="macro", labels=CLASSES, zero_division=0)
print(f"majority class={maj} | macro-F1(val)={f1_maj:.3f}")


## 6. Fine-tune ViT (freeze backbone, train head)


In [ ]:
model = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=len(CLASSES))
for p in model.parameters():
    p.requires_grad = False
for p in model.head.parameters():
    p.requires_grad = True
model = model.to(DEVICE)

opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
crit = nn.CrossEntropyLoss()


def run_epoch(loader, train: bool):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    preds, trues = [], []
    for x, y, *_ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = crit(logits, y)
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
        loss_sum += loss.item() * len(y)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
        preds.extend(pred.detach().cpu().tolist())
        trues.extend(y.detach().cpu().tolist())
    acc = correct / total
    f1 = f1_score(trues, preds, average="macro", zero_division=0)
    return loss_sum / total, acc, f1


history = {"train_f1": [], "val_f1": []}
for ep in range(1, EPOCHS_HEAD + 1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    history["train_f1"].append(tr[2])
    history["val_f1"].append(va[2])
    print(f"epoch {ep:02d} | train F1={tr[2]:.3f} acc={tr[1]:.3f} | val F1={va[2]:.3f} acc={va[1]:.3f}")

print(f"\nbaseline macro-F1={f1_maj:.3f} | ViT val macro-F1={history['val_f1'][-1]:.3f}")
print("ViT beats baseline?" , history["val_f1"][-1] > f1_maj)


## 7. Curves + confusion matrix


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["train_f1"], label="train F1")
plt.plot(history["val_f1"], label="val F1")
plt.axhline(f1_maj, color="gray", ls="--", label=f"majority ({f1_maj:.2f})")
plt.xlabel("epoch")
plt.ylabel("macro-F1")
plt.legend()
plt.title("ViT fine-tune (head only)")
plt.tight_layout()
plt.savefig(FIG / "vit_f1_curves.png", dpi=120)
plt.show()

# confusion on val
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for x, y, *_ in val_loader:
        pred = model(x.to(DEVICE)).argmax(1).cpu().tolist()
        all_pred.extend(pred)
        all_true.extend(y.tolist())

cm = confusion_matrix(all_true, all_pred, labels=list(range(len(CLASSES))))
print(classification_report(all_true, all_pred, target_names=CLASSES, zero_division=0))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels([c.split("_")[0] for c in CLASSES], rotation=45, ha="right")
ax.set_yticklabels([c.split("_")[0] for c in CLASSES])
ax.set_xlabel("predicted")
ax.set_ylabel("true")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(FIG / "confusion.png", dpi=120)
plt.show()


## 8. JSON contract (`vision.extract`)

Schema is fixed **before** agent answers. Invalid JSON = pipeline failure.


In [ ]:
SCHEMA_FIELDS = ["type", "confidence", "ticker", "fields", "source_tool"]


def predict_one(path: Path):
    model.eval()
    x = tf_eval(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, 1)[0]
        conf, idx = float(prob.max()), int(prob.argmax())
    return CLASSES[idx], conf


def vision_extract(path: Path, ticker_hint: str | None) -> dict:
    cls, conf = predict_one(path)
    # ticker comes from a naive "OCR" stub: we use the hint stored at generation time
    payload = {
        "type": cls,
        "confidence": round(conf, 3),
        "ticker": ticker_hint if ticker_hint else None,
        "fields": {"title": path.stem, "period": "2024"},
        "source_tool": "vision.extract",
    }
    missing = [k for k in SCHEMA_FIELDS if k not in payload]
    if missing:
        raise ValueError(f"invalid JSON, missing {missing}")
    return payload


# demo one val sample with ticker and one without
with_t = next(s for s in val_s if s.ticker)
without_t = next(s for s in val_s if not s.ticker)
print("WITH ticker:")
print(json.dumps(vision_extract(with_t.path, with_t.ticker), indent=2))
print("\nNULL ticker:")
print(json.dumps(vision_extract(without_t.path, without_t.ticker), indent=2))


## 9. MCP tool stubs

Same four tools as the case. `market.*` / `knowledge.*` are deterministic stubs.


In [ ]:
MARKET = {
    "AAPL": {"name": "Apple Inc.", "currency": "USD", "sector": "Technology"},
    "MSFT": {"name": "Microsoft Corp.", "currency": "USD", "sector": "Technology"},
    "GOOGL": {"name": "Alphabet Inc.", "currency": "USD", "sector": "Technology"},
    "AMZN": {"name": "Amazon.com Inc.", "currency": "USD", "sector": "Consumer"},
}

GLOSSARY = {
    "candlestick": "OHLC bars used in technical charting.",
    "line_chart": "Price path as a continuous line.",
    "balance_sheet": "Snapshot of assets, liabilities, equity.",
}


def vision_classify(path: Path) -> dict:
    cls, conf = predict_one(path)
    return {"class": cls, "score": round(conf, 3)}


def market_lookup(ticker: str | None) -> dict:
    if not ticker:
        return {"error": "null_ticker"}
    return MARKET.get(ticker, {"error": "unknown_ticker", "ticker": ticker})


def knowledge_search(query: str) -> dict:
    for k, v in GLOSSARY.items():
        if k in query.lower().replace(" ", "_") or k in query.lower():
            return {"query": query, "snippet": v}
    return {"query": query, "snippet": "No entry (stub glossary)."}


print("tools ready:", ["vision.classify", "vision.extract", "market.lookup", "knowledge.search"])


## 10. Grounded answer vs refusal

Requirement from the case:
1. Vision assertion  
2. Tool justification  
3. Refusal when tool fails / null


In [ ]:
def grounded_answer(sample: Sample) -> dict:
    """Simulate an agent that only speaks from tools."""
    trace = []
    c = vision_classify(sample.path)
    trace.append({"tool": "vision.classify", "out": c})
    ex = vision_extract(sample.path, sample.ticker)
    trace.append({"tool": "vision.extract", "out": ex})
    m = market_lookup(ex.get("ticker"))
    trace.append({"tool": "market.lookup", "out": m})
    k = knowledge_search(c["class"])
    trace.append({"tool": "knowledge.search", "out": k})

    if ex.get("ticker") is None or "error" in m:
        answer = (
            "I cannot confirm a ticker: extraction returned null or market.lookup failed. "
            "I refuse to invent a symbol. Retry with a wider crop / different DPI."
        )
        status = "refusal"
    else:
        answer = (
            f"vision.classify labels this as {c['class']} (score {c['score']}). "
            f"vision.extract returns ticker={ex['ticker']}. "
            f"market.lookup: {m.get('name')} ({m.get('currency')}). "
            f"I assert no closing price: no price field is present in the JSON."
        )
        status = "grounded"
    return {"status": status, "answer": answer, "trace": trace}


g = grounded_answer(with_t)
r = grounded_answer(without_t)

print("=== GROUNDED ===")
print(g["status"])
print(g["answer"])
print(json.dumps(g["trace"], indent=2)[:800], "...")

print("\n=== REFUSAL ===")
print(r["status"])
print(r["answer"])

(OUT / "grounded_trace.json").write_text(json.dumps(g, indent=2))
(OUT / "refusal_trace.json").write_text(json.dumps(r, indent=2))
print("\nsaved traces in", OUT)


## 11. Save metrics for the 4-page note


In [ ]:
metrics = {
    "n_train": len(train_s),
    "n_val": len(val_s),
    "n_val_docs": len(val_docs),
    "majority_macro_f1": round(f1_maj, 4),
    "vit_val_macro_f1": round(history["val_f1"][-1], 4),
    "beats_baseline": bool(history["val_f1"][-1] > f1_maj),
    "model": "vit_tiny_patch16_224 (head fine-tune)",
    "split": "by document",
    "tools": ["vision.classify", "vision.extract", "market.lookup", "knowledge.search"],
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


## Checklist (before oral)

- [ ] Document-level split (no page leakage)
- [ ] ViT beats (or honestly fails) majority baseline
- [ ] Confusion matrix + 5 errors discussed
- [ ] JSON schema written before agent
- [ ] Grounded trace saved
- [ ] Refusal after null ticker / tool KO saved
- [ ] Three limits written (layout, capture/DPI, hallucination)
